In [1]:
import os, zipfile

# Укажи точный путь к архиву на твоем диске
zip_path = '/content/drive/MyDrive/Colab Notebooks/archive.zip'
extract_path = '/content/dataset'

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_path)

print("✅ Распаковка завершена")
print("Содержимое:", os.listdir(extract_path))

✅ Распаковка завершена
Содержимое: ['new plant diseases dataset(augmented)', 'New Plant Diseases Dataset(Augmented)', 'test']


In [2]:
!pip install timm pandas torch torchvision -q

import timm, torch, torch.nn as nn, torch.optim as optim
import pandas as pd, time, os
from torchvision import transforms, datasets
from torch.utils.data import DataLoader

# 1. НАСТРОЙКИ (ЗДЕСЬ МЕНЯЕМ ЭПОХИ)
NUM_EPOCHS = 5  # <-- Ограничение в количестве эпох
BATCH_SIZE = 32 # <-- При оибке CUDA out of memory уменьшить до 24 или 16 (на крайняк)
IMG_SIZE = 224
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# 2. ДАННЫЕ (Укажи правильный путь к папке с train и val)
data_dir = '/content/dataset/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)' # Путь к твоим данным
transform_train = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
transform_val = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_ds = datasets.ImageFolder(f'{data_dir}/train', transform=transform_train)
val_ds = datasets.ImageFolder(f'{data_dir}/valid', transform=transform_val)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# 3. СПИСОК МОДЕЛЕЙ
models_to_train = ['resnet18', 'efficientnet_b0', 'mobilenetv3_large_100', 'densenet121', 'vit_base_patch16_224']
results = []

# 4. ЦИКЛ ОБУЧЕНИЯ
for model_name in models_to_train:
    print(f"\n--- Запуск {model_name} ---")

    # Загрузка модели
    model = timm.create_model(model_name, pretrained=True, num_classes=len(train_ds.classes)).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=1e-4)

    # Обучение (строго NUM_EPOCHS раз)
    start_time = time.time()
    model.train()
    for epoch in range(NUM_EPOCHS):
        running_loss = 0.0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        print(f"Epoch {epoch+1}/{NUM_EPOCHS} | Loss: {running_loss/len(train_loader):.4f}")

    # Замер скорости инференса
    model.eval()
    infer_start = time.time()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs = imgs.to(device)
            preds = model(imgs).argmax(dim=1)
            correct += (preds == labels.to(device)).sum().item()
            total += labels.size(0)
    infer_time = time.time() - infer_start

    acc = correct / total
    size_mb = sum(p.numel() for p in model.parameters()) * 4 / (1024**2)

    results.append({
        'Model': model_name,
        'Epochs': NUM_EPOCHS,
        'Val Accuracy': round(acc, 4),
        'Infer Time (s)': round(infer_time, 2),
        'Size MB': round(size_mb, 1)
    })

    # Сохранение лучшей модели (для приложения)
    torch.save(model.state_dict(), f'/content/drive/MyDrive/dataset/best_{model_name}.pth')
    print(f"Сохранено: best_{model_name}.pth")
    torch.cuda.empty_cache()

# 5. ИТОГОВАЯ ТАБЛИЦА
df = pd.DataFrame(results)
print("\nИтоговое сравнение:")
print(df)
df.to_csv('/content/drive/MyDrive/dataset/training_results.csv', index=False)


--- Запуск resnet18 ---


model.safetensors:   0%|          | 0.00/46.8M [00:00<?, ?B/s]

Epoch 1/5 | Loss: 0.7678
Epoch 2/5 | Loss: 0.0778
Epoch 3/5 | Loss: 0.0399
Epoch 4/5 | Loss: 0.0243
Epoch 5/5 | Loss: 0.0168
Сохранено: best_resnet18.pth

--- Запуск efficientnet_b0 ---


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

Epoch 1/5 | Loss: 0.1827
Epoch 2/5 | Loss: 0.0186
Epoch 3/5 | Loss: 0.0124
Epoch 4/5 | Loss: 0.0094
Epoch 5/5 | Loss: 0.0091
Сохранено: best_efficientnet_b0.pth

--- Запуск mobilenetv3_large_100 ---


...

Epoch 1/5 | Loss: 0.1845
Epoch 2/5 | Loss: 0.0221
Epoch 3/5 | Loss: 0.0136
Epoch 4/5 | Loss: 0.0102
Epoch 5/5 | Loss: 0.0110
Сохранено: best_mobilenetv3_large_100.pth

--- Запуск densenet121 ---


model.safetensors:   0%|          | 0.00/32.3M [00:00<?, ?B/s]

Epoch 1/5 | Loss: 0.2529
Epoch 2/5 | Loss: 0.0286
Epoch 3/5 | Loss: 0.0175
Epoch 4/5 | Loss: 0.0143
Epoch 5/5 | Loss: 0.0097
Сохранено: best_densenet121.pth

--- Запуск vit_base_patch16_224 ---


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Epoch 1/5 | Loss: 0.1498
Epoch 2/5 | Loss: 0.0561
Epoch 3/5 | Loss: 0.0456
Epoch 4/5 | Loss: 0.0357
Epoch 5/5 | Loss: 0.0357
Сохранено: best_vit_base_patch16_224.pth

Итоговое сравнение:
                   Model  Epochs  Val Accuracy  Infer Time (s)  Size MB
0               resnet18       5        0.9946           24.61     42.7
1        efficientnet_b0       5        0.9957           27.45     15.5
2  mobilenetv3_large_100       5        0.9956           24.92     16.2
3            densenet121       5        0.9965           31.91     26.7
4   vit_base_patch16_224       5        0.9822           90.81    327.4


In [6]:
import torch
import timm
from torchvision import transforms
from PIL import Image
import os
import shutil
import re
from collections import defaultdict

# Загружаем модель
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = timm.create_model('efficientnet_b0', pretrained=False, num_classes=38).to(device)
model.load_state_dict(torch.load('/content/drive/MyDrive/dataset/best_efficientnet_b0.pth', map_location=device))
model.eval()

# Получаем список классов из модели (они должны совпадать с train/val)
train_dir = '/content/dataset/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/train'
classes = sorted([d.name for d in os.scandir(train_dir) if d.is_dir()])
print(f"Найдено {len(classes)} классов в обучающей выборке")
print("Примеры классов:", classes[:3])

# Сканируем папку test
test_dir = '/content/dataset/test/test'
test_files = [f for f in os.listdir(test_dir) if f.endswith(('.JPG', '.jpg', '.png'))]
print(f"\nНайдено {len(test_files)} файлов в test")

# Автоматический маппинг: извлекаем префикс из имени файла
def extract_prefix(filename):
    """Извлекает префикс из имени файла, убирая цифры и расширение"""
    # Убираем расширение
    name = os.path.splitext(filename)[0]
    # Убираем цифры в конце
    prefix = re.sub(r'\d+$', '', name)
    return prefix

def split_camel_case(text):
    """Разбивает CamelCase на список слов"""
    # Вставляем пробел перед заглавной буквой
    words = re.sub(r'([A-Z])', r' \1', text).split()
    return [w.lower() for w in words if w]

def find_class_for_file(prefix, classes):
    """Находит класс, соответствующий префиксу файла"""
    # Разбиваем префикс на слова
    prefix_words = split_camel_case(prefix)

    # Ищем класс, который содержит все слова из префикса
    for cls in classes:
        cls_lower = cls.lower()
        # Проверяем, что все слова из префикса есть в названии класса
        if all(word in cls_lower for word in prefix_words):
            return cls

    return None

# Создаём маппинг файлов -> классы
file_class_mapping = {}
unmapped_files = []

for filename in test_files:
    prefix = extract_prefix(filename)
    true_class = find_class_for_file(prefix, classes)

    if true_class:
        file_class_mapping[filename] = true_class
    else:
        unmapped_files.append((filename, prefix))

print(f"\n✅ Сопоставлено файлов: {len(file_class_mapping)}")
if unmapped_files:
    print(f"⚠️ Не удалось сопоставить: {len(unmapped_files)}")
    for fname, prefix in unmapped_files[:5]:
        print(f"  {fname} -> префикс: {prefix}")

# Трансформация
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Очищаем папки для примеров
shutil.rmtree('/content/correct_examples', ignore_errors=True)
shutil.rmtree('/content/error_examples', ignore_errors=True)
os.makedirs('/content/correct_examples', exist_ok=True)
os.makedirs('/content/error_examples', exist_ok=True)

correct_examples = []
error_examples = []

# Проходим по всем файлам с известным классом
print(f"\n🔍 Анализируем {len(file_class_mapping)} изображений...")

for filename, true_class in file_class_mapping.items():
    img_path = os.path.join(test_dir, filename)

    try:
        img = Image.open(img_path).convert('RGB')
        inputs = transform(img).unsqueeze(0).to(device)

        with torch.no_grad():
            outputs = model(inputs)
            probs = torch.nn.functional.softmax(outputs[0], dim=0)
            top_prob, top_idx = probs.topk(1)
            pred_class = classes[top_idx.item()]
            confidence = top_prob.item() * 100

        # Собираем примеры
        if pred_class == true_class:
            if len(correct_examples) < 3:
                correct_examples.append({
                    'file': filename,
                    'true': true_class,
                    'pred': pred_class,
                    'conf': confidence
                })
                shutil.copy(img_path, f'/content/correct_examples/{filename}')
        else:
            if len(error_examples) < 3:
                error_examples.append({
                    'file': filename,
                    'true': true_class,
                    'pred': pred_class,
                    'conf': confidence
                })
                shutil.copy(img_path, f'/content/error_examples/{filename}')

    except Exception as e:
        print(f"Ошибка обработки {filename}: {e}")

# Вывод результатов
print("\n" + "="*60)
print("=== РЕЗУЛЬТАТЫ ===")
print(f"Всего обработано: {len(file_class_mapping)}")
print(f"Правильно: {len(correct_examples)}")
print(f"Ошибок: {len(error_examples)}")
print("="*60)

print("\n=== УДАЧНЫЕ ПРИМЕРЫ ===")
for i, ex in enumerate(correct_examples, 1):
    print(f"{i}. {ex['file']}: {ex['true']} (уверенность {ex['conf']:.2f}%)")

print("\n=== ОШИБОЧНЫЕ ПРИМЕРЫ ===")
if error_examples:
    for i, ex in enumerate(error_examples, 1):
        print(f"{i}. {ex['file']}")
        print(f"   True: {ex['true']}")
        print(f"   Pred: {ex['pred']} (уверенность {ex['conf']:.2f}%)")
else:
    print("❌ Ошибки не найдены! Модель работает идеально на этом наборе.")
    print("\n💡 Попробуем найти примеры с низкой уверенностью...")

    # Ищем предсказания с низкой уверенностью
    low_conf_examples = []
    for filename, true_class in file_class_mapping.items():
        img_path = os.path.join(test_dir, filename)
        img = Image.open(img_path).convert('RGB')
        inputs = transform(img).unsqueeze(0).to(device)

        with torch.no_grad():
            outputs = model(inputs)
            probs = torch.nn.functional.softmax(outputs[0], dim=0)
            top_prob, top_idx = probs.topk(1)
            pred_class = classes[top_idx.item()]
            confidence = top_prob.item() * 100

        if confidence < 95 and pred_class != true_class:
            low_conf_examples.append({
                'file': filename,
                'true': true_class,
                'pred': pred_class,
                'conf': confidence
            })

    if low_conf_examples:
        print(f"\nНайдено {len(low_conf_examples)} примеров с уверенностью < 95%:")
        for ex in low_conf_examples[:3]:
            print(f"  {ex['file']}: True={ex['true']}, Pred={ex['pred']} ({ex['conf']:.2f}%)")

print("\n✅ Примеры сохранены:")
print(f"   /content/correct_examples/ — {len(correct_examples)} файлов")
print(f"   /content/error_examples/ — {len(error_examples)} файлов")

Найдено 38 классов в обучающей выборке
Примеры классов: ['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust']

Найдено 33 файлов в test

✅ Сопоставлено файлов: 33

🔍 Анализируем 33 изображений...

=== РЕЗУЛЬТАТЫ ===
Всего обработано: 33
Правильно: 3
Ошибок: 0

=== УДАЧНЫЕ ПРИМЕРЫ ===
1. PotatoEarlyBlight3.JPG: Potato___Early_blight (уверенность 100.00%)
2. PotatoEarlyBlight2.JPG: Potato___Early_blight (уверенность 100.00%)
3. TomatoHealthy3.JPG: Tomato___healthy (уверенность 100.00%)

=== ОШИБОЧНЫЕ ПРИМЕРЫ ===
❌ Ошибки не найдены! Модель работает идеально на этом наборе.

💡 Попробуем найти примеры с низкой уверенностью...

✅ Примеры сохранены:
   /content/correct_examples/ — 3 файлов
   /content/error_examples/ — 0 файлов


In [9]:
import torch, timm, os, shutil, re
from torchvision import transforms
from PIL import Image, ImageEnhance, ImageFilter

# === ПУТИ ===
TRAIN_DIR = '/content/dataset/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/train'
TEST_DIR = '/content/dataset/test/test'
MODEL_PATH = '/content/drive/MyDrive/dataset/best_efficientnet_b0.pth'

# === 1. АВТОМАТИЧЕСКОЕ ОПРЕДЕЛЕНИЕ КЛАССОВ ===
classes = sorted([d.name for d in os.scandir(TRAIN_DIR) if d.is_dir()])
print(f"Найдено {len(classes)} классов:")
for i, c in enumerate(classes):
    print(f"  {i}: {c}")

# === 2. ЗАГРУЗКА МОДЕЛИ ===
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = timm.create_model('efficientnet_b0', pretrained=False, num_classes=len(classes)).to(device)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()
print(f"\n✅ Модель загружена ({device})")

# === 3. АВТОМАТИЧЕСКИЙ МАППИНГ ФАЙЛОВ В КЛАССЫ ===
test_files = [f for f in os.listdir(TEST_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
print(f"Найдено {len(test_files)} тестовых изображений")

def extract_prefix(filename):
    name = os.path.splitext(filename)[0]
    prefix = re.sub(r'\d+$', '', name)
    return prefix

def split_camel(text):
    words = re.sub(r'([A-Z])', r' \1', text).split()
    return [w.lower() for w in words if w]

def find_class(prefix, classes):
    words = split_camel(prefix)
    for cls in classes:
        cls_lower = cls.lower().replace('___', ' ').replace('_', ' ')
        if all(word in cls_lower for word in words):
            return cls
    # Попытка частичного совпадения (первые 2 слова)
    if len(words) >= 2:
        for cls in classes:
            cls_lower = cls.lower().replace('___', ' ').replace('_', ' ')
            if all(word in cls_lower for word in words[:2]):
                return cls
    return None

mapping = {}
unmapped = []
for f in test_files:
    prefix = extract_prefix(f)
    cls = find_class(prefix, classes)
    if cls:
        mapping[f] = cls
    else:
        unmapped.append(f)

print(f"\n✅ Сопоставлено: {len(mapping)}")
if unmapped:
    print(f"⚠️ Не сопоставлено: {len(unmapped)}")
    for f in unmapped[:5]:
        print(f"   {f} -> {extract_prefix(f)}")

# === 4. ТРАНСФОРМАЦИЯ ===
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

def predict(img):
    inputs = transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        outputs = model(inputs)
        probs = torch.nn.functional.softmax(outputs[0], dim=0)
        top_prob, top_idx = probs.topk(1)
        return classes[top_idx.item()], top_prob.item() * 100

# === 5. ПОИСК УДАЧНЫХ И ОШИБОЧНЫХ ПРИМЕРОВ ===
out_correct = '/content/correct_examples'
out_error = '/content/error_examples'
for d in [out_correct, out_error]:
    shutil.rmtree(d, ignore_errors=True)
    os.makedirs(d, exist_ok=True)

correct = []
errors = []

# Сначала ищем ошибки на оригиналах
for fname, true_cls in mapping.items():
    img_path = os.path.join(TEST_DIR, fname)
    try:
        img = Image.open(img_path).convert('RGB')
        pred_cls, conf = predict(img)

        if pred_cls != true_cls:
            errors.append({'file': fname, 'type': 'Оригинал', 'true': true_cls, 'pred': pred_cls, 'conf': conf})
            shutil.copy(img_path, f'{out_error}/{fname}')
        else:
            correct.append({'file': fname, 'true': true_cls, 'pred': pred_cls, 'conf': conf})
    except Exception as e:
        print(f"Ошибка: {fname}: {e}")

print(f"\nНа оригиналах: ошибок={len(errors)}, правильных={len(correct)}")

# Если ошибок мало — стресс-тесты
if len(errors) < 3:
    print("\n🔍 Запуск стресс-тестов для поиска ошибок...")

    stress_tests = [
        ("dark_20", lambda img: ImageEnhance.Brightness(img).enhance(0.2)),
        ("dark_40", lambda img: ImageEnhance.Brightness(img).enhance(0.4)),
        ("blur_3", lambda img: img.filter(ImageFilter.GaussianBlur(radius=3))),
        ("blur_7", lambda img: img.filter(ImageFilter.GaussianBlur(radius=7))),
        ("contrast_20", lambda img: ImageEnhance.Contrast(img).enhance(0.2)),
        ("contrast_40", lambda img: ImageEnhance.Contrast(img).enhance(0.4)),
        ("color_20", lambda img: ImageEnhance.Color(img).enhance(0.2)),
        ("sharpness_10", lambda img: ImageEnhance.Sharpness(img).enhance(0.1)),
    ]

    test_subset = list(mapping.items())[:30]  # Берём первые 30 изображений

    for fname, true_cls in test_subset:
        if len(errors) >= 5:
            break
        img_path = os.path.join(TEST_DIR, fname)
        try:
            img = Image.open(img_path).convert('RGB')

            for test_name, transform_fn in stress_tests:
                if len(errors) >= 5:
                    break

                stress_img = transform_fn(img)
                pred_cls, conf = predict(stress_img)

                if pred_cls != true_cls:
                    save_name = f"{test_name}_{fname}"
                    stress_img.save(f'{out_error}/{save_name}')
                    errors.append({
                        'file': save_name,
                        'type': test_name,
                        'true': true_cls,
                        'pred': pred_cls,
                        'conf': conf
                    })
        except Exception as e:
            pass

# Сохраняем 3 удачных примера
for ex in correct[:3]:
    src = os.path.join(TEST_DIR, ex['file'])
    dst = os.path.join(out_correct, ex['file'])
    if os.path.exists(src):
        shutil.copy(src, dst)

# === 6. ВЫВОД РЕЗУЛЬТАТОВ ===
print("\n" + "=" * 60)
print("=== УДАЧНЫЕ ПРИМЕРЫ ===")
for i, ex in enumerate(correct[:3], 1):
    print(f"{i}. {ex['file']}")
    print(f"   True: {ex['true']}")
    print(f"   Pred: {ex['pred']} (уверенность {ex['conf']:.2f}%)")

print("\n=== ОШИБОЧНЫЕ ПРИМЕРЫ ===")
for i, ex in enumerate(errors[:3], 1):
    print(f"{i}. {ex['file']}")
    print(f"   Тип: {ex['type']}")
    print(f"   True: {ex['true']}")
    print(f"   Pred: {ex['pred']} (уверенность {ex['conf']:.2f}%)")

print("\n" + "=" * 60)
print(f"✅ Файлы сохранены:")
print(f"   {out_correct}/ — {len(os.listdir(out_correct))} файлов")
print(f"   {out_error}/ — {len(os.listdir(out_error))} файлов")

Найдено 38 классов:
  0: Apple___Apple_scab
  1: Apple___Black_rot
  2: Apple___Cedar_apple_rust
  3: Apple___healthy
  4: Blueberry___healthy
  5: Cherry_(including_sour)___Powdery_mildew
  6: Cherry_(including_sour)___healthy
  7: Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot
  8: Corn_(maize)___Common_rust_
  9: Corn_(maize)___Northern_Leaf_Blight
  10: Corn_(maize)___healthy
  11: Grape___Black_rot
  12: Grape___Esca_(Black_Measles)
  13: Grape___Leaf_blight_(Isariopsis_Leaf_Spot)
  14: Grape___healthy
  15: Orange___Haunglongbing_(Citrus_greening)
  16: Peach___Bacterial_spot
  17: Peach___healthy
  18: Pepper,_bell___Bacterial_spot
  19: Pepper,_bell___healthy
  20: Potato___Early_blight
  21: Potato___Late_blight
  22: Potato___healthy
  23: Raspberry___healthy
  24: Soybean___healthy
  25: Squash___Powdery_mildew
  26: Strawberry___Leaf_scorch
  27: Strawberry___healthy
  28: Tomato___Bacterial_spot
  29: Tomato___Early_blight
  30: Tomato___Late_blight
  31: Tomato___Leaf

In [8]:
import torch
import timm
from torchvision import transforms
from PIL import Image, ImageEnhance, ImageFilter
import os
import shutil
import re

# === ПУТИ ===
TRAIN_DIR = '/content/dataset/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/train'
TEST_DIR = '/content/dataset/test/test'
MODEL_PATH = '/content/drive/MyDrive/dataset/best_efficientnet_b0.pth'
OUT_CORRECT = '/content/correct_examples'
OUT_ERROR = '/content/error_examples'

# === ЗАГРУЗКА МОДЕЛИ ===
device = 'cuda' if torch.cuda.is_available() else 'cpu'
classes = sorted([d.name for d in os.scandir(TRAIN_DIR) if d.is_dir()])
print(f"✅ Найдено классов: {len(classes)}")
print(f"   Примеры: {classes[:3]}")

model = timm.create_model('efficientnet_b0', pretrained=False, num_classes=len(classes)).to(device)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()
print(f"✅ Модель загружена: {MODEL_PATH}")

# === АВТОМАППИНГ ИМЁН ФАЙЛОВ В КЛАССЫ ===
def split_camel_case(text):
    return [w.lower() for w in re.sub(r'([A-Z])', r' \1', text).split() if w]

def find_class_for_prefix(prefix, classes):
    words = split_camel_case(prefix)
    for cls in classes:
        cls_lower = cls.lower().replace('___', ' ').replace('_', ' ')
        if all(w in cls_lower for w in words):
            return cls
    return None

def extract_prefix(filename):
    return re.sub(r'\d+$', '', os.path.splitext(filename)[0])

# === ПОДГОТОВКА ===
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

for d in [OUT_CORRECT, OUT_ERROR]:
    shutil.rmtree(d, ignore_errors=True)
    os.makedirs(d, exist_ok=True)

# === СКАНИРОВАНИЕ TEST ===
test_files = [f for f in os.listdir(TEST_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
print(f"\n🔍 Файлов в test: {len(test_files)}")

correct_examples, error_examples = [], []
stats = {'total': 0, 'correct': 0, 'error': 0, 'unmapped': 0}

for filename in test_files:
    prefix = extract_prefix(filename)
    true_class = find_class_for_prefix(prefix, classes)

    if true_class is None:
        stats['unmapped'] += 1
        continue

    img_path = os.path.join(TEST_DIR, filename)
    try:
        img = Image.open(img_path).convert('RGB')
        inputs = transform(img).unsqueeze(0).to(device)
        with torch.no_grad():
            outputs = model(inputs)
            probs = torch.nn.functional.softmax(outputs[0], dim=0)
            top_prob, top_idx = probs.topk(1)
            pred_class = classes[top_idx.item()]
            conf = top_prob.item() * 100

        stats['total'] += 1
        if pred_class == true_class:
            stats['correct'] += 1
            if len(correct_examples) < 3:
                correct_examples.append({'file': filename, 'true': true_class, 'pred': pred_class, 'conf': conf})
                shutil.copy(img_path, os.path.join(OUT_CORRECT, filename))
        else:
            stats['error'] += 1
            if len(error_examples) < 3:
                error_examples.append({'file': filename, 'true': true_class, 'pred': pred_class, 'conf': conf})
                shutil.copy(img_path, os.path.join(OUT_ERROR, filename))
    except Exception as e:
        print(f"⚠️ Ошибка {filename}: {e}")

# === РЕЗУЛЬТАТЫ ===
print("\n" + "="*60)
print(f"Обработано: {stats['total']} | Верно: {stats['correct']} | Ошибок: {stats['error']} | Не маппится: {stats['unmapped']}")
print("="*60)

print("\n✅ УДАЧНЫЕ ПРИМЕРЫ:")
for i, ex in enumerate(correct_examples, 1):
    print(f"{i}. {ex['file']}")
    print(f"   True: {ex['true']}")
    print(f"   Pred: {ex['pred']} ({ex['conf']:.2f}%)")

print("\n❌ ОШИБОЧНЫЕ ПРИМЕРЫ:")
for i, ex in enumerate(error_examples, 1):
    print(f"{i}. {ex['file']}")
    print(f"   True: {ex['true']}")
    print(f"   Pred: {ex['pred']} ({ex['conf']:.2f}%)")

# === ПЛАН Б: СТРЕСС-ТЕСТЫ, ЕСЛИ ОШИБОК НЕТ ===
if len(error_examples) < 3 and correct_examples:
    print("\n⚠️ Модель слишком точная. Запускаю стресс-тесты...")
    stress_cases = []

    for ex in correct_examples[:3]:
        img_path = os.path.join(TEST_DIR, ex['file'])
        img = Image.open(img_path).convert('RGB')

        # Три типа искажений
        distortions = {
            'dark': ImageEnhance.Brightness(img).enhance(0.2),
            'blur': img.filter(ImageFilter.GaussianBlur(radius=6)),
            'low_contrast': ImageEnhance.Contrast(img).enhance(0.3)
        }

        for name, distorted in distortions.items():
            if len(error_examples) >= 3:
                break
            distorted_path = os.path.join(OUT_ERROR, f"{name}_{ex['file']}")
            distorted.save(distorted_path)

            inputs = transform(distorted).unsqueeze(0).to(device)
            with torch.no_grad():
                outputs = model(inputs)
                probs = torch.nn.functional.softmax(outputs[0], dim=0)
                top_prob, top_idx = probs.topk(1)
                pred_class = classes[top_idx.item()]
                conf = top_prob.item() * 100

            if pred_class != ex['true'] or conf < 85:
                error_examples.append({
                    'file': f"{name}_{ex['file']}",
                    'true': ex['true'],
                    'pred': pred_class,
                    'conf': conf,
                    'distortion': name
                })
                print(f"   ➕ Стресс-тест {name}: {ex['true']} → {pred_class} ({conf:.1f}%)")

    if len(error_examples) >= 3:
        print(f"\n✅ Получено {len(error_examples)} примеров через стресс-тесты")

print(f"\n📁 Файлы сохранены:")
print(f"   {OUT_CORRECT}/ — {len(os.listdir(OUT_CORRECT))} шт.")
print(f"   {OUT_ERROR}/ — {len(os.listdir(OUT_ERROR))} шт.")

✅ Найдено классов: 38
   Примеры: ['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust']
✅ Модель загружена: /content/drive/MyDrive/dataset/best_efficientnet_b0.pth

🔍 Файлов в test: 33

Обработано: 33 | Верно: 33 | Ошибок: 0 | Не маппится: 0

✅ УДАЧНЫЕ ПРИМЕРЫ:
1. PotatoEarlyBlight3.JPG
   True: Potato___Early_blight
   Pred: Potato___Early_blight (100.00%)
2. PotatoEarlyBlight2.JPG
   True: Potato___Early_blight
   Pred: Potato___Early_blight (100.00%)
3. TomatoHealthy3.JPG
   True: Tomato___healthy
   Pred: Tomato___healthy (100.00%)

❌ ОШИБОЧНЫЕ ПРИМЕРЫ:

⚠️ Модель слишком точная. Запускаю стресс-тесты...
   ➕ Стресс-тест dark: Potato___Early_blight → Tomato___Early_blight (89.0%)
   ➕ Стресс-тест blur: Potato___Early_blight → Blueberry___healthy (29.8%)
   ➕ Стресс-тест dark: Potato___Early_blight → Potato___Early_blight (76.6%)

✅ Получено 3 примеров через стресс-тесты

📁 Файлы сохранены:
   /content/correct_examples/ — 3 шт.
   /content/error_examples/ — 4 шт.


In [10]:
import os
import json

# Путь к папке train
TRAIN_DIR = '/content/dataset/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/train'

# Автоматически получаем список классов
classes = sorted([d.name for d in os.scandir(TRAIN_DIR) if d.is_dir()])

print(f"Найдено {len(classes)} классов:")
for i, cls in enumerate(classes):
    print(f"  {i}: {cls}")

# Сохраняем в JSON
output_path = '/content/classes.json'
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(classes, f, ensure_ascii=False, indent=2)

print(f"\n✅ Список классов сохранён в {output_path}")

Найдено 38 классов:
  0: Apple___Apple_scab
  1: Apple___Black_rot
  2: Apple___Cedar_apple_rust
  3: Apple___healthy
  4: Blueberry___healthy
  5: Cherry_(including_sour)___Powdery_mildew
  6: Cherry_(including_sour)___healthy
  7: Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot
  8: Corn_(maize)___Common_rust_
  9: Corn_(maize)___Northern_Leaf_Blight
  10: Corn_(maize)___healthy
  11: Grape___Black_rot
  12: Grape___Esca_(Black_Measles)
  13: Grape___Leaf_blight_(Isariopsis_Leaf_Spot)
  14: Grape___healthy
  15: Orange___Haunglongbing_(Citrus_greening)
  16: Peach___Bacterial_spot
  17: Peach___healthy
  18: Pepper,_bell___Bacterial_spot
  19: Pepper,_bell___healthy
  20: Potato___Early_blight
  21: Potato___Late_blight
  22: Potato___healthy
  23: Raspberry___healthy
  24: Soybean___healthy
  25: Squash___Powdery_mildew
  26: Strawberry___Leaf_scorch
  27: Strawberry___healthy
  28: Tomato___Bacterial_spot
  29: Tomato___Early_blight
  30: Tomato___Late_blight
  31: Tomato___Leaf

In [11]:
import streamlit as st
import torch
import timm
import json
import os
from datetime import datetime
from torchvision import transforms
from PIL import Image

# Настройка страницы
st.set_page_config(page_title="Диагностика болезней растений", layout="centered")

st.title("🌿 Диагностика болезней растений")
st.markdown("**Система компьютерного зрения для анализа состояния листьев**")

# Предупреждение об ограничениях
st.warning("⚠️ **Внимание:** Модель является учебным прототипом. Не используйте для реальной агрономической диагностики без дополнительной проверки!")

# === ЗАГРУЗКА СПИСКА КЛАССОВ ===
CLASSES_FILE = 'classes.json'

if not os.path.exists(CLASSES_FILE):
    st.error(f" Файл {CLASSES_FILE} не найден. Запустите скрипт генерации классов из датасета.")
    st.stop()

with open(CLASSES_FILE, 'r', encoding='utf-8') as f:
    CLASSES = json.load(f)

st.sidebar.success(f"✅ Загружено {len(CLASSES)} классов")

# === ЗАГРУЗКА МОДЕЛИ ===
MODEL_PATH = 'best_efficientnet_b0.pth'  # Положи рядом с app.py

@st.cache_resource
def load_model():
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = timm.create_model('efficientnet_b0', pretrained=False, num_classes=len(CLASSES))
    try:
        model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    except Exception as e:
        st.error(f"Не удалось загрузить модель: {e}")
        st.stop()
    model.to(device)
    model.eval()
    return model, device

try:
    model, device = load_model()
    st.success(f"✅ Модель загружена: EfficientNet-B0 ({device})")
except Exception as e:
    st.error(f"Ошибка загрузки модели: {e}")
    st.stop()

# Трансформация
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# === БОКОВАЯ ПАНЕЛЬ СО СТАТИСТИКОЙ ===
st.sidebar.header("📊 Статистика")
HISTORY_FILE = "history.json"

if os.path.exists(HISTORY_FILE):
    with open(HISTORY_FILE, "r", encoding="utf-8") as f:
        history = json.load(f)
    st.sidebar.metric("Всего проверок", len(history))

    if history:
        disease_counts = {}
        for h in history:
            disease = h.get('result', 'Unknown')
            disease_counts[disease] = disease_counts.get(disease, 0) + 1

        st.sidebar.subheader("Топ заболеваний")
        for disease, count in sorted(disease_counts.items(), key=lambda x: x[1], reverse=True)[:5]:
            st.sidebar.text(f"{disease[:30]}: {count}")
else:
    history = []
    st.sidebar.info("История пуста")

# === ЗАГРУЗКА ИЗОБРАЖЕНИЯ ===
st.subheader("📷 Загрузка изображения")
uploaded_file = st.file_uploader(
    "Загрузите фотографию листа",
    type=["jpg", "jpeg", "png"],
    help="Поддерживаемые форматы: JPG, PNG"
)

if uploaded_file is not None:
    col1, col2 = st.columns(2)

    with col1:
        img = Image.open(uploaded_file).convert('RGB')
        st.image(img, caption="Загруженное изображение", use_column_width=True)

    if st.button("🔍 Провести диагностику", type="primary"):
        with st.spinner("Анализ изображения..."):
            try:
                inputs = transform(img).unsqueeze(0).to(device)
                with torch.no_grad():
                    outputs = model(inputs)
                    probs = torch.nn.functional.softmax(outputs[0], dim=0)

                top_probs, top_classes = probs.topk(3)

                with col2:
                    st.subheader(" Результаты диагностики")

                    top_class_name = CLASSES[top_classes[0].item()]
                    top_confidence = top_probs[0].item() * 100

                    st.success(f"**Диагноз:** {top_class_name}")
                    st.metric("Уверенность модели", f"{top_confidence:.2f}%")

                    if 'healthy' in top_class_name.lower():
                        st.info("✅ Растение здорово")
                    else:
                        disease_name = top_class_name.split('___')[-1].replace('_', ' ')
                        st.error(f"️ Обнаружено заболевание: **{disease_name}**")

                    st.markdown("**Топ-3 предсказания:**")
                    for i in range(3):
                        cls_name = CLASSES[top_classes[i].item()]
                        conf = top_probs[i].item() * 100
                        st.text(f"{i+1}. {cls_name} — {conf:.2f}%")

                # Сохранение в историю
                result_record = {
                    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                    "file": uploaded_file.name,
                    "result": CLASSES[top_classes[0].item()],
                    "confidence": f"{top_probs[0].item()*100:.2f}%",
                    "top3": [
                        {
                            "class": CLASSES[top_classes[i].item()],
                            "confidence": f"{top_probs[i].item()*100:.2f}%"
                        }
                        for i in range(3)
                    ]
                }

                history.append(result_record)
                with open(HISTORY_FILE, "w", encoding="utf-8") as f:
                    json.dump(history, f, ensure_ascii=False, indent=2)

                # Экспорт в CSV
                st.markdown("---")
                st.subheader("📥 Экспорт результатов")

                csv_data = "Дата,Файл,Результат,Уверенность\n"
                for h in history:
                    csv_data += f"{h['timestamp']},{h['file']},{h['result']},{h['confidence']}\n"

                st.download_button(
                    label="📥 Скачать историю (CSV)",
                    data=csv_data,
                    file_name=f"diagnosis_report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv",
                    mime="text/csv"
                )

                # Ограничения модели
                with st.expander("⚠️ Ограничения модели"):
                    st.markdown("""
                    **Модель может ошибаться в следующих случаях:**
                    - Плохое освещение (темные или пересвеченные фото)
                    - Размытые изображения
                    - Частично закрытые листья
                    - Сложный фон
                    - Похожими заболеваниями разных культур

                    **Рекомендации:**
                    - Делайте фото при хорошем освещении
                    - Фокусируйтесь на листе
                    - Используйте простой фон
                    """)

            except Exception as e:
                st.error(f"Ошибка при анализе: {e}")

else:
    st.info("👆 Загрузите изображение для начала диагностики")

# Footer
st.markdown("---")
st.markdown(
    "**Разработано в рамках учебной практики по компьютерному зрению** | "
    "МТУСИ, Кафедра «Программная инженерия» | 2026"
)

ModuleNotFoundError: No module named 'streamlit'